# Sparkfun Crawled EAGLE / KiCad Zip link

In [ ]:
import csv
import time
import requests
from urllib.parse import urlparse, parse_qs, urlunparse, urlencode, urljoin
from lxml import html
from requests.adapters import HTTPAdapter, Retry

# ---------- HTTP session with retries ----------
def _session_with_retries(total=3, backoff=0.5, timeout=20):
    s = requests.Session()
    retries = Retry(
        total=total,
        backoff_factor=backoff,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['HEAD','GET','OPTIONS'])
    )
    s.mount('http://', HTTPAdapter(max_retries=retries))
    s.mount('https://', HTTPAdapter(max_retries=retries))
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/124.0.0.0 Safari/537.36")
    })
    s.request_timeout = timeout
    return s

# ---------- Utilities ----------
def _normalize_to_page1(url: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    qs.pop('p', None)
    return urlunparse(parsed._replace(query=urlencode({k: v[0] for k, v in qs.items()})))

def _is_empty_page(tree) -> bool:
    """
    Detect Magento 'no products' message:
    <div class="message info empty"><div>We can't find products matching the selection.</div></div>
    """
    return bool(tree.xpath("//div[contains(@class,'message') and contains(@class,'info') and contains(@class,'empty')]"))

def _iter_pages_until_empty(category_url: str, sess: requests.Session, max_pages: int = 200):
    """
    Yield page URLs from ?p=1 increasing until an empty page is encountered,
    or until max_pages is reached (safety cap).
    """
    base_url = _normalize_to_page1(category_url)
    parsed = urlparse(base_url)

    for i in range(1, max_pages + 1):
        qs = parse_qs(parsed.query)
        qs['p'] = [str(i)]
        page_url = urlunparse(parsed._replace(query=urlencode({k: v[0] for k, v in qs.items()})))

        r = sess.get(page_url, timeout=sess.request_timeout)
        r.raise_for_status()
        tree = html.fromstring(r.content)

        if _is_empty_page(tree):
            break  # stop when Magento shows the empty message

        yield page_url

def _get_product_links_from_page(page_url: str, sess: requests.Session) -> list[str]:
    r = sess.get(page_url, timeout=sess.request_timeout)
    r.raise_for_status()
    tree = html.fromstring(r.content)

    anchors = tree.xpath('//*[@id="amasty-shopby-product-list"]/div[2]/ol//a[@href]')
    seen, out = set(), []
    for a in anchors:
        href = a.get('href')
        if href:
            absolute = urljoin(page_url, href)
            if absolute not in seen:
                seen.add(absolute)
                out.append(absolute)
    return out

def _find_kicad_eagle_links(product_url: str, sess: requests.Session):
    """
    Return (kicad_url, eagle_url)
    - kicad_url: .zip whose anchor text contains 'kicad'
    - eagle_url: .zip whose anchor text contains 'eagle', else 3rd .zip found if exists
    """
    r = sess.get(product_url, timeout=sess.request_timeout)
    r.raise_for_status()
    tree = html.fromstring(r.content)

    zip_links_in_order = []
    kicad_url = ""
    eagle_url = ""

    for a in tree.xpath('//a[@href]'):
        href = (a.get('href') or '').strip()
        if not href.lower().endswith('.zip'):
            continue
        abs_url = urljoin(product_url, href)
        zip_links_in_order.append(abs_url)

        text = (a.text_content() or '').strip().lower()
        if not kicad_url and "kicad" in text:
            kicad_url = abs_url
        if not eagle_url and "eagle" in text:
            eagle_url = abs_url

    # Deduplicate while preserving order
    seen = set()
    zip_links_in_order = [u for u in zip_links_in_order if not (u in seen or seen.add(u))]

    # Eagle fallback
    if not eagle_url and len(zip_links_in_order) >= 3:
        eagle_url = zip_links_in_order[2]

    return kicad_url, eagle_url

# ---------- Main Function ----------
def category_to_csv(category_url: str, csv_path: str, delay_sec: float = 0.0):
    """
    Crawl category URL and save CSV with columns:
    page_url, product_url, kicad_url, eagle_url
    Iterates pages sequentially (?p=1,2,3,...) until the 'empty' message appears.
    """
    sess = _session_with_retries()

    page_count = 0
    product_count = 0

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["page_url", "product_url", "kicad_url", "eagle_url"])

        for page_url in _iter_pages_until_empty(category_url, sess):
            page_count += 1
            try:
                product_links = _get_product_links_from_page(page_url, sess)
                product_count += len(product_links)
                print(f"[Page {page_count}] {page_url} -> {len(product_links)} products")
            except Exception as e:
                print(f"[Page error] {page_url} -> {e}")
                continue

            if delay_sec:
                time.sleep(delay_sec)

            for product_url in product_links:
                try:
                    kicad_url, eagle_url = _find_kicad_eagle_links(product_url, sess)
                except Exception as e:
                    print(f"[Product error] {product_url} -> {e}")
                    kicad_url, eagle_url = "", ""

                writer.writerow([page_url, product_url, kicad_url, eagle_url])

                if delay_sec:
                    time.sleep(delay_sec)

    print(f"Done. Pages crawled: {page_count}, products seen: {product_count}")


def csv_path_from_url(url, base_folder):
    """Convert category URL to CSV path inside base_folder."""
    path_name = os.path.basename(urlparse(url).path)  # e.g., "components.html"
    if path_name.endswith(".html"):
        path_name = path_name[:-5]  # remove ".html"
    file_name = f"sparkfun_{path_name}_zips.csv"
    return os.path.join(base_folder, file_name)

# Example

In [ ]:
sparkfun_category_url =['https://www.sparkfun.com/audio.html',
                        'https://www.sparkfun.com/components.html',
                        'https://www.sparkfun.com/data-logging-and-memory.html',
                        'https://www.sparkfun.com/development-boards.html',
                        'https://www.sparkfun.com/displays.html',
                        'https://www.sparkfun.com/e-textiles-crafting.html',
                        'https://www.sparkfun.com/gnss.html',
                        'https://www.sparkfun.com/iot-wireless.html',
                        'https://www.sparkfun.com/kits.html',
                        'https://www.sparkfun.com/robotics.html',
                        'https://www.sparkfun.com/sensors.html',
                        'https://www.sparkfun.com/tools.html',
                        'https://www.sparkfun.com/special-categories.html']

In [ ]:
base_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"
cat_url = 'https://www.sparkfun.com/data-logging-and-memory.html'
category_to_csv(
    cat_url,
    csv_path_from_url(cat_url, base_folder),
    delay_sec=0.2        # polite throttling
)

In [ ]:
base_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"

for cat_url in sparkfun_category_url:
    category_to_csv(
        cat_url,
        csv_path_from_url(cat_url, base_folder),
        delay_sec=0.2        # polite throttling
    )

# Download Zip 

In [ ]:
import os
import csv
from collections import Counter
import pathlib
import requests
from urllib.parse import urlparse

def combine_csv_files(src_folder, output_csv):
    """
    Combine all CSV files in a folder into a single CSV file.
    Assumes all CSVs have the same header.

    Args:
        src_folder (str): Path to folder containing CSV files
        output_csv (str): Path to save the combined CSV
    """
    csv_files = [f for f in os.listdir(src_folder) if f.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("No CSV files found in the source folder.")

    header_written = False

    with open(output_csv, "w", newline="", encoding="utf-8") as out_f:
        writer = None

        for csv_file in csv_files:
            csv_path = os.path.join(src_folder, csv_file)
            with open(csv_path, newline="", encoding="utf-8") as in_f:
                reader = csv.reader(in_f)
                header = next(reader)

                if not header_written:
                    writer = csv.writer(out_f)
                    writer.writerow(header)
                    header_written = True

                for row in reader:
                    writer.writerow(row)

    print(f"Combined {len(csv_files)} CSV files into: {output_csv}")




def get_zip_links_from_csv(csv_path, url_col_index=3):
    """
    Get a list of ZIP file links from a specific column of a CSV file.

    Args:
        csv_path (str): Path to the CSV file.
        url_col_index (int): Zero-based index of the column containing the URLs.
                             Default is 3 (fourth column).

    Returns:
        list[str]: List of ZIP file URLs from the specified column.
    """
    zip_links = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader, None)  # skip header
        for row in reader:
            if len(row) > url_col_index:
                url = row[url_col_index].strip()
                if url.lower().endswith(".zip"):
                    zip_links.append(url)
    return zip_links




def find_duplicate_links(links):
    """
    Find duplicate links in a list.

    Args:
        links (list[str]): List of links

    Returns:
        list[str]: List of duplicate links (each appears once in output)
    """
    counter = Counter(links)
    duplicates = [link for link, count in counter.items() if count > 1]
    return duplicates


def remove_duplicates(links):
    """
    Remove duplicate links from a list while preserving order.

    Args:
        links (list[str]): List of links

    Returns:
        list[str]: List with duplicates removed
    """
    seen = set()
    unique_links = []
    for link in links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)
    return unique_links


def download_zips_to_named_folders(links, dest_folder, overwrite=False, timeout=30):
    """
    Download ZIP files from a list of URLs and save each into its own subfolder.
    Folder name is based on the ZIP name (without extension) from the URL.
    The ZIP file inside is always saved as 'eagle.zip'.
    If folder exists and overwrite=False, add ##1, ##2... until unique.

    Args:
        links (list[str]): List of ZIP URLs
        dest_folder (str): Destination base folder
        overwrite (bool): Overwrite if file exists
        timeout (int): Timeout for download in seconds

    Returns:
        tuple: (list of saved paths, list of failed URLs)
    """
    os.makedirs(dest_folder, exist_ok=True)
    saved_paths = []
    failed_urls = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }

    total = len(links)
    for idx, url in enumerate(links, start=1):
        try:
            # Base folder name from ZIP file name (without .zip)
            base_folder_name = pathlib.Path(urlparse(url).path).stem or f"file_{idx}"

            # Ensure unique folder if overwrite=False
            zip_folder = os.path.join(dest_folder, base_folder_name)
            if not overwrite:
                counter = 1
                while os.path.exists(zip_folder):
                    zip_folder = os.path.join(dest_folder, f"{base_folder_name}##{counter}")
                    counter += 1

            os.makedirs(zip_folder, exist_ok=True)
            dest_path = os.path.join(zip_folder, "eagle.zip")

            print(f"[{idx}/{total}] Downloading: {url} -> {dest_path}")

            with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                tmp_path = dest_path + ".part"
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 64):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp_path, dest_path)

            saved_paths.append(dest_path)

        except Exception as e:
            print(f"[{idx}/{total}] Failed to download {url}: {e}")
            failed_urls.append(url)

    # Summary report
    print("\n=== Download Summary ===")
    print(f"✅ Successful downloads: {len(saved_paths)}")
    print(f"❌ Failed downloads: {len(failed_urls)}")
    if failed_urls:
        print("Failed URLs:")
        for u in failed_urls:
            print("  -", u)

    return saved_paths, failed_urls



# Example

In [ ]:
src_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"
output_csv = "/Users/taitinglu/Desktop/IMG2SCH/sparkfun_combined.csv"
dest_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip"

# combine_csv_files(src_folder, output_csv)

In [ ]:
links = get_zip_links_from_csv(output_csv)
duplicates = find_duplicate_links(links)
print(len(duplicates))
unique_links = remove_duplicates(links)
print("Unique links:", len(unique_links))
download_zips_to_named_folders(unique_links, dest_folder)

# Sparkfun Tutorial

In [ ]:
import csv
import requests
from lxml import html

UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

def _class_contains(x):
    return f"contains(concat(' ', normalize-space(@class), ' '), ' {x} ')"

def save_sparkfun_titles_and_urls(page_url, csv_path):
    """
    Scrape ONLY grid tiles from the given SparkFun tutorials page
    and save their title + URL into a CSV file.

    Args:
        page_url (str): URL of the SparkFun tutorials page (e.g., "?page=all")
        csv_path (str): Output CSV file path
    """
    r = requests.get(page_url, headers={"User-Agent": UA}, timeout=25)
    r.raise_for_status()

    tree = html.fromstring(r.content)

    container_xpath = f"//*[@id='airlock']//div[{_class_contains('tutorials-index')}]"
    containers = tree.xpath(container_xpath)
    if not containers:
        print("[WARN] tutorials-index container not found.")
        return
    container = containers[0]

    tiles_xpath = (
        f".//div[{_class_contains('tile')} and {_class_contains('tutorial-tile')} and {_class_contains('grid')}]"
    )
    tiles = container.xpath(tiles_xpath)

    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Title", "URL"])  # header

        for t in tiles:
            # Title
            title_nodes = t.xpath(".//h3[contains(@class,'title')]/text()")
            title = title_nodes[0].strip() if title_nodes else ""

            # URL
            a = t.xpath(".//a[@href]")
            url_t = a[0].get("href") if a else ""

            writer.writerow([title, url_t])

    print(f"[OK] Saved {len(tiles)} tutorials to {csv_path}")






In [ ]:
tutorial_url = "https://learn.sparkfun.com/tutorials?page=all"
tutorial_csv = '/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials.csv'
save_sparkfun_titles_and_urls(
    tutorial_url,
    tutorial_csv
)

In [ ]:
import csv
import re
import time
from typing import Tuple
from urllib.parse import urljoin

import requests
from lxml import html
from requests.adapters import HTTPAdapter, Retry

UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

def _session_with_retries(total=3, backoff=0.5, timeout=25) -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=total,
        backoff_factor=backoff,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["HEAD", "GET", "OPTIONS"]),
        raise_on_status=False,
    )
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update({"User-Agent": UA, "Accept": "text/html,application/xhtml+xml"})
    s.request_timeout = timeout
    return s

def _fetch_html(session: requests.Session, url: str) -> str:
    resp = session.get(url, timeout=getattr(session, "request_timeout", 25))
    resp.raise_for_status()
    if not resp.encoding or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    return resp.text

def _extract_zip_links(html_text: str, base_url: str) -> Tuple[str, str]:
    tree = html.fromstring(html_text)
    anchors = tree.xpath("//a[@href]")
    zip_re = re.compile(r"\.zip(?:[?#].*)?$", re.IGNORECASE)

    eagle_url = ""
    kicad_url = ""

    for a in anchors:
        text = (a.text_content() or "").strip()
        href = a.get("href", "")
        if not href:
            continue
        abs_url = urljoin(base_url, href)
        if not zip_re.search(abs_url):
            continue

        low = text.lower()
        if not eagle_url and "eagle" in low:
            eagle_url = abs_url
        if not kicad_url and "kicad" in low:
            kicad_url = abs_url

        if eagle_url and kicad_url:
            break

    return eagle_url, kicad_url

def enrich_tutorial_csv_with_design_files(
    input_csv_path: str,
    output_csv_path: str,
    delay_sec: float = 0.2
):
    """
    Read CSV with headers Title,URL and write a new CSV with:
    Title,URL,eagle_zip_url,kicad_zip_url
    Prints an index while processing.
    """
    session = _session_with_retries()

    with open(input_csv_path, "r", encoding="utf-8-sig", newline="") as infile, \
         open(output_csv_path, "w", encoding="utf-8", newline="") as outfile:

        reader = list(csv.DictReader(infile))  # convert to list so we can get total length
        total = len(reader)

        fieldnames = ["Title", "URL", "eagle_zip_url", "kicad_zip_url"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        for idx, row in enumerate(reader, start=1):
            title = row.get("Title", "").strip()
            url = row.get("URL", "").strip()
            eagle_zip = ""
            kicad_zip = ""

            print(f"[{idx}/{total}] Processing: {title or '(no title)'}")

            if url:
                try:
                    html_text = _fetch_html(session, url)
                    eagle_zip, kicad_zip = _extract_zip_links(html_text, url)
                except Exception as e:
                    print(f"  [WARN] Failed to fetch/extract for {url} -> {e}")
                time.sleep(delay_sec)

            writer.writerow({
                "Title": title,
                "URL": url,
                "eagle_zip_url": eagle_zip,
                "kicad_zip_url": kicad_zip
            })

    print(f"[OK] Wrote enriched CSV to: {output_csv_path}")





# Example

In [ ]:
# Example usage:
if __name__ == "__main__":
    enrich_tutorial_csv_with_design_files(
        input_csv_path="/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials.csv",          # must have Title,URL
        output_csv_path="/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials_with_zips.csv",
        delay_sec=0.25
    )

In [ ]:
output_csv = '/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials_with_zips.csv'
dest_folder = '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Tutorial Zip'

links = get_zip_links_from_csv(output_csv,url_col_index=2)
duplicates = find_duplicate_links(links)
print("duplicate: ",len(duplicates))
unique_links = remove_duplicates(links)
print("Unique links:", len(unique_links))
download_zips_to_named_folders(unique_links, dest_folder)

# Check Two Image file are same

In [2]:
import os
import xml.etree.ElementTree as ET

def xml_files_equal(
    path1: str,
    path2: str,
    *,
    mode: str = "semantic",              # "semantic" or "byte"
    ignore_whitespace: bool = True,      # collapse whitespace in text nodes
    ignore_comments: bool = True,        # drop <!-- comments -->
    ignore_child_order: bool = False     # if True, treat children as an unordered set
):
    """
    Compare two XML files.

    Returns:
        (equal: bool, detail: str)
    """
    if mode not in {"semantic", "byte"}:
        raise ValueError("mode must be 'semantic' or 'byte'")

    if mode == "byte":
        # Byte-for-byte equality (fast, simplest)
        try:
            with open(path1, "rb") as f1, open(path2, "rb") as f2:
                b1, b2 = f1.read(), f2.read()
            return (b1 == b2, "byte-equal" if b1 == b2 else "byte-different")
        except Exception as e:
            return (False, f"I/O error: {e}")

    # ---- semantic comparison below ----
    def norm_text(s: str | None) -> str:
        if s is None:
            return ""
        if ignore_whitespace:
            # collapse internal whitespace and strip edges
            return " ".join(s.split())
        return s

    def is_comment(node) -> bool:
        # xml.etree uses a factory for comments; safest check via tag string
        return isinstance(node.tag, str) and node.tag is ET.Comment  # rarely True in stdlib
        # Fallback (works reliably):
        # return ET.iselement(node) and node.tag == ET.Comment

    def canonical(elem):
        """
        Convert an Element into a canonical, hashable tuple representation.
        """
        # tag includes namespace in '{uri}local' form when present
        tag = elem.tag

        # attributes: compare as sorted tuple of key/value (keys include namespaces)
        attrs = tuple(sorted((k, norm_text(v)) for k, v in elem.attrib.items()))

        # normalized text; ignore .tail completely (usually formatting)
        text = norm_text(elem.text)

        # children: optionally drop comments and normalize order
        children = []
        for child in list(elem):
            # xml.etree doesn't always keep Comment nodes unless explicitly parsed;
            # handle anyway if present.
            if ignore_comments and child.tag is ET.Comment:
                continue
            children.append(canonical(child))

        if ignore_child_order:
            children.sort()

        return (tag, attrs, text, tuple(children))

    try:
        tree1 = ET.parse(path1)
        tree2 = ET.parse(path2)
    except Exception as e:
        return (False, f"XML parse error: {e}")

    c1 = canonical(tree1.getroot())
    c2 = canonical(tree2.getroot())

    return (c1 == c2, "semantic-equal" if c1 == c2 else "semantic-different")


# Optional: if lxml is installed, you can use C14N (canonical XML) for robust comparison
def xml_files_equal_c14n(path1: str, path2: str):
    """
    Compare two XML files using XML C14N (requires lxml).
    Returns (equal: bool, detail: str). Falls back with a message if lxml not available.
    """
    try:
        from lxml import etree
    except ImportError:
        return (False, "lxml not installed for C14N")

    try:
        parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
        t1 = etree.parse(path1, parser)
        t2 = etree.parse(path2, parser)

        c14n1 = etree.tostring(t1, method="c14n")
        c14n2 = etree.tostring(t2, method="c14n")
        return (c14n1 == c14n2, "c14n-equal" if c14n1 == c14n2 else "c14n-different")
    except Exception as e:
        return (False, f"C14N error: {e}")


In [4]:
file1 = r'/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Github Sch UnZip/LilyPad_LED_Blue_(5pcs).sch'
file2 = r'/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Github Sch UnZip/LilyPad_LED_Green_(5pcs).sch'
ok, detail = xml_files_equal(file1, file2)  # semantic by default
print(ok, detail)

True semantic-equal
